In [1]:
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import entropy


In [2]:

def compute_frequency_entropy(window, fs=256):
    """Compute Frequency Entropy for all EEG channels in a given window."""
    frequency_entropies = []
    
    for i in range(window.shape[1]):  # Iterate over EEG channels
        f, Pxx = welch(window[:, i], fs=fs, nperseg=1024, window='hann', scaling='density')
        
        # Normalize Power Spectrum
        Pxx_norm = Pxx / np.sum(Pxx)
        
        # Compute Shannon Entropy
        freq_entropy = entropy(Pxx_norm)  # Apply entropy formula
        
        frequency_entropies.append(freq_entropy)
    
    return frequency_entropies


In [3]:

def sliding_window_frequency_entropy(eeg_data, outcomes, eeg_columns, window_size, step_size, fs=256):
    """Extract Frequency Entropy features using a sliding window approach."""
    all_frequency_entropy_features = []
    targets = []
    n_samples = eeg_data.shape[0]
    
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window = eeg_data[start:end]
        outcome_window = outcomes[start:end]

        # Compute Frequency Entropy features for this window
        freq_entropy_values = compute_frequency_entropy(window, fs)

        all_frequency_entropy_features.append(freq_entropy_values)
        targets.append(1 if np.any(outcome_window) else 0)  # Assign target based on Outcome
    
    return np.array(all_frequency_entropy_features), np.array(targets)


In [4]:

# Main processing
if __name__ == "__main__":
    # Load EEG data
    eeg_data_path = '/Users/puchku-home/Study/PROJECT/EEG/EEG Assets/chbmit_preprocessed_data.csv'
    data = pd.read_csv(eeg_data_path)
    eeg_columns = [col for col in data.columns if col != 'Outcome']

    # Convert to NumPy arrays
    eeg_data = np.asarray(data[eeg_columns].values, dtype=np.float32)
    outcomes = np.asarray(data['Outcome'].values, dtype=np.float32)

    # Windowing parameters
    fs = 256  # Sampling frequency
    window_size = fs * 1  # 1-second windows
    step_size = window_size // 2  # 50% overlap

    # Compute Frequency Entropy features
    freq_entropy_features, targets = sliding_window_frequency_entropy(eeg_data, outcomes, eeg_columns, window_size, step_size, fs)

    # Convert to DataFrame and save
    freq_entropy_feature_names = [f"{col}_freq_entropy" for col in eeg_columns]
    
    freq_entropy_df = pd.DataFrame(freq_entropy_features, columns=freq_entropy_feature_names)
    freq_entropy_df['target'] = targets
    
    output_file_path = '/Users/puchku-home/Downloads/Frequency Feature  Generalised/Frequency_Entropy_Features.csv'
    freq_entropy_df.to_csv(output_file_path, index=False)
    
    print(f"✅ Frequency Entropy feature extraction complete. Features saved to '{output_file_path}'")


/Users/puchku-home/Study/PROJECT/EEG/myeeg/lib/python3.13/site-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 1024 is greater than input length  = 256, using nperseg = 256
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
/var/folders/ht/lcgnytjx5js3222vs1k678s80000gn/T/ipykernel_7000/1849332450.py:9: RuntimeWarning: invalid value encountered in divide
  Pxx_norm = Pxx / np.sum(Pxx)


✅ Frequency Entropy feature extraction complete. Features saved to '/Users/puchku-home/Downloads/Frequency Feature  Generalised/Frequency_Entropy_Features.csv'
